In [1]:
import sys
import subprocess
sys.path.insert(0, "../Src/")

import numpy as np
import pandas as pd
import os
from IPython.display import display
import json
from os.path import exists
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import backend as K

%autosave 5

Autosaving every 5 seconds


In [2]:
def loadData(dataset, split=0, batch_size=32, seed=51):
    mfccFiles = {
        'training': '../Data/mfcc/mfcc_train.npz',
        'testing': '../Data/mfcc/mfcc_test.npz'
    }

    mfccPath = mfccFiles.get(dataset, dataset)

    if not exists(mfccPath):
        print(f'{mfccPath} was not found. Regenerating MFCC features...')
        import createMFCCs as cmfcc
        cmfcc.createMFCCMatricies(output_dir='../Data/mfcc')

    if not exists(mfccPath):
        raise FileNotFoundError(f'Could not find MFCC archive: {mfccPath}')

    with np.load(mfccPath) as mfccData:
        X = mfccData['X'].astype(np.float32)
        y = mfccData['y'].astype(np.float32)

    if len(X) == 0:
        raise ValueError(f'No MFCC samples were found in {mfccPath}')

    if X.ndim == 3:
        X = np.expand_dims(X, -1)

    rng = np.random.default_rng(seed)
    shuffledIndices = rng.permutation(len(X))
    X = X[shuffledIndices]
    y = y[shuffledIndices]

    def makeDataset(features, labels, shouldShuffle=True):
        dataset = tf.data.Dataset.from_tensor_slices((features, labels))
        if shouldShuffle and len(features) > 1:
            dataset = dataset.shuffle(len(features), seed=seed, reshuffle_each_iteration=True)
        return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    if split == 0:
        print('In non split')
        return makeDataset(X, y)

    print('In split')
    if len(X) < 2:
        raise ValueError(f'Need at least 2 samples to split {mfccPath}')

    splitIndex = max(1, int(len(X) * split))
    splitIndex = min(splitIndex, len(X) - 1)

    devX = X[:splitIndex]
    devY = y[:splitIndex]
    testX = X[splitIndex:]
    testY = y[splitIndex:]

    return makeDataset(devX, devY, shouldShuffle=False), makeDataset(testX, testY, shouldShuffle=False)

In [3]:
def f1_score(y_true, y_pred): #taken from old keras source code 
    y_true = K.round(K.clip(y_true, 0, 1))
    y_pred = K.round(K.clip(y_pred, 0, 1))
    
    true_positives = K.sum(y_true * y_pred)
    predicted_positives = K.sum(y_pred)
    possible_positives = K.sum(y_true)
    
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    
    f1_val = 2 * (precision * recall) / (precision + recall + K.epsilon())
    return f1_val

In [4]:
def createModel(convFilters1, convFilters2, convFilters3, convFilters4, numberOfFCLayers, numberOfNeuronsPerFCLayer, adamLearningRate, L2Rate):
    model = keras.Sequential()
        
    #convPool1
    model.add(keras.layers.Conv2D(convFilters1,(3,3), activation='relu',padding='valid'))
    model.add(keras.layers.MaxPooling2D(pool_size=(2,2), padding='valid'))
    #convPool2
    model.add(keras.layers.Conv2D(convFilters2,(3,3), activation='relu',padding='valid'))
    model.add(keras.layers.MaxPooling2D(pool_size=(2,2), padding='valid'))
    #convPool3
    model.add(keras.layers.Conv2D(convFilters3,(3,3), activation='relu',padding='valid'))
    model.add(keras.layers.MaxPooling2D(pool_size=(2,2), padding='valid'))
    #finalConv
    model.add(keras.layers.Conv2D(convFilters4,(3,3), activation='relu',padding='valid'))
    
    model.add(keras.layers.Flatten())
    
    for layer in range(numberOfFCLayers):
        if layer == numberOfFCLayers - 1:
            model.add(keras.layers.Dense(1,activation='sigmoid'))
        else:
            model.add(keras.layers.Dense(numberOfNeuronsPerFCLayer,activation='relu',kernel_regularizer=tf.keras.regularizers.l2(L2Rate)))

    try:
        adamOptimizer = keras.optimizers.legacy.Adam(learning_rate=adamLearningRate)
    except AttributeError:
        adamOptimizer = keras.optimizers.Adam(learning_rate=adamLearningRate)

    model.compile(optimizer=adamOptimizer,loss='binary_crossentropy', metrics=[f1_score])
    return model

In [5]:
def createModelParametersDF(n_convFilters1,n_convFilters2,n_convFilters3,n_convFilters4,
                            n_FCLayers,n_NeuronsPerFCLayers,n_Epochs,adamLearningRates,
                            L2Rates,trainScores,devScores):
    modelParameters = dict()
    modelParameters['n_convFilters1'] = n_convFilters1
    modelParameters['n_convFilters2'] = n_convFilters2
    modelParameters['n_convFilters3'] = n_convFilters3
    modelParameters['n_convFilters4'] = n_convFilters4
    modelParameters['n_FCLayers'] = n_FCLayers
    modelParameters['n_NeuronsPerFCLayers'] = n_NeuronsPerFCLayers
    modelParameters['n_Epochs'] = n_Epochs
    modelParameters['adamLearningRates'] = adamLearningRates
    modelParameters['L2Rates'] = L2Rates
    modelParameters['trainScore'] = trainScores
    modelParameters['devScore'] = devScores

    modelParametersDF = pd.DataFrame(modelParameters, columns=modelParameters.keys())
    return modelParametersDF

In [6]:
def createRangeFromMidpoint(midpoint,range,mandatoryMinimum=1):
    possibleMin = int(midpoint-(range/2))
    possibleMin = max([mandatoryMinimum,possibleMin])
    possibleMax = int(midpoint+(range/2))
    possibleRange = np.arange(possibleMin,possibleMax)
    return possibleRange


def generateAdamLearningRate(minVal=1e-4,maxVal=1e-2):
    minVal = np.log10(minVal)
    maxVal = np.log10(maxVal)
    learningRatePower = np.random.random() * (maxVal - minVal) + minVal
    learningRate = np.power(10,learningRatePower)
    return learningRate


def generateL2(minVal=1e-2,maxVal=1e3):
    minVal = np.log10(minVal)
    maxVal = np.log10(maxVal)
    l2Power = np.random.random() * (maxVal - minVal) + minVal
    l2 = np.power(10,l2Power)
    return l2


def calculateCriticalPoints(top5ParamList):
    lowPoint = np.min(top5ParamList)
    highPoint = np.max(top5ParamList)
    return (lowPoint,highPoint)


def calculateLogisticCriticalPoints(top5ParamList):
    top5Log10ParamList = np.log10(top5ParamList)
    log10criticalPointTuple = calculateCriticalPoints(top5Log10ParamList)
    criticalPointTuple = (np.power(10,log10criticalPointTuple[0]),np.power(10,log10criticalPointTuple[1]))
    return criticalPointTuple


def getAdjustedRange(top5ParamList):
    lowerValue = int(np.max([1,np.min(top5ParamList)]))
    upperValue = int(np.max(top5ParamList))
    
    if lowerValue == upperValue:
        return createRangeFromMidpoint(lowerValue,2*lowerValue)
    return np.arange(lowerValue,upperValue)


def displayFinalResults(parameterFileName):
    if (exists(parameterFileName)):
        with open(parameterFileName) as d:
            finalResults = json.load(d)
            resultsDictionary = dict()
            for key in finalResults.keys():
                resultsDictionary[key] = [finalResults[key]]
            resultsDF = pd.DataFrame(resultsDictionary,columns = finalResults.keys())
            print('Final Model')
            display(resultsDF)

In [7]:
def main():

    possibleConvFilters1 = createRangeFromMidpoint(32,64)
    possibleConvFilters2 = createRangeFromMidpoint(32,64)
    possibleConvFilters3 = createRangeFromMidpoint(32,64)
    possibleConvFilters4 = createRangeFromMidpoint(32,64)

    possibleNumberOfFCLayers = createRangeFromMidpoint(10,20)
    possibleNumberOfNeuronsPerFCLayer = createRangeFromMidpoint(10,20)

    possibleNumberOfEpochs = createRangeFromMidpoint(10,20)
    dropoutCriticalPoints = (0,1)
    adamLearningRateCriticalPoints = (1e-4,1e-2)
    L2CriticalPoints = (1e-2,1e3)    
    
    trial = 0
    bestDevScore = 0

    os.makedirs('../Models', exist_ok=True)
    os.makedirs('../Models/DetectVoicesModelTrials', exist_ok=True)
    
    train = loadData('training')
    dev,test = loadData('testing',.5)

    choose = np.random.choice

    n_convFilters1 = []
    n_convFilters2 = []
    n_convFilters3 = []
    n_convFilters4 = []
    n_FCLayers = []
    n_NeuronsPerFCLayers = []
    n_Epochs = []
    adamLearningRates = []
    L2Rates = []
    trainScores = []
    devScores = []
    
    while trial < 100:
        convFilters1 = choose(possibleConvFilters1)
        convFilters2 = choose(possibleConvFilters2)
        convFilters3 = choose(possibleConvFilters3)
        convFilters4 = choose(possibleConvFilters4)

        numberOfFCLayers = choose(possibleNumberOfFCLayers)
        numberOfNeuronsPerFCLayer = choose(possibleNumberOfNeuronsPerFCLayer)

        numberOfEpochs = choose(possibleNumberOfEpochs)
        
        adamLearningRate = generateAdamLearningRate(adamLearningRateCriticalPoints[0],adamLearningRateCriticalPoints[1])
        L2Rate = generateL2(L2CriticalPoints[0],L2CriticalPoints[1])
        

        model = createModel(convFilters1, convFilters2, convFilters3, convFilters4,
                            numberOfFCLayers, numberOfNeuronsPerFCLayer, adamLearningRate,L2Rate)
    
        model.fit(train,epochs=numberOfEpochs,verbose=0)

        model_path = f'../Models/DetectVoicesModelTrials/dv_model_{trial}.h5'
        model.save(model_path)
        model_size = os.path.getsize(model_path) / (1024 * 1024)
        if model_size < 40:
            print()
            print('trainScore')
            trainScore = float(model.evaluate(train, verbose=0)[1])
            print(trainScore)
            print('devScore')
            devScore = float(model.evaluate(dev, verbose=0)[1])
            print(devScore)

            if (devScore > 0.91) and (devScore > bestDevScore):
                testScore = float(model.evaluate(test, verbose=0)[1])
                model_path = f'../Models/best_dv_model_.h5'
                model.save(model_path)
                bestModelParams = {
                    'n_convFilters1' : int(convFilters1),
                    'n_convFilters2' : int(convFilters2),
                    'n_convFilters3' : int(convFilters3),
                    'n_convFilters4' : int(convFilters4),
                    'n_FCLayers' : int(numberOfFCLayers),
                    'n_NeuronsPerFCLayers' : int(numberOfNeuronsPerFCLayer),
                    'n_Epochs' : int(numberOfEpochs),
                    'adamLearningRates' : float(adamLearningRate),
                    'L2Rates' : float(L2Rate),
                    'modelSize' : float(model_size),
                    'trainScore': float(trainScore),
                    'devScore': float(devScore),
                    'testScore': float(testScore)
                }
                with open('../Models/best_dv_model_params.json', 'w') as f:
                    json.dump(bestModelParams, f)
                bestDevScore = devScore       

            n_convFilters1.append(int(convFilters1))
            n_convFilters2.append(int(convFilters2))
            n_convFilters3.append(int(convFilters3))
            n_convFilters4.append(int(convFilters4))
            
            n_FCLayers.append(int(numberOfFCLayers))
            n_NeuronsPerFCLayers.append(int(numberOfNeuronsPerFCLayer))
            n_Epochs.append(int(numberOfEpochs))
            
            adamLearningRates.append(float(adamLearningRate))
            L2Rates.append(float(L2Rate))
            trainScores.append(float(trainScore))
            devScores.append(float(devScore))
            
            print('concluding trial ',trial)
            trial += 1
        else:
            print(f'redoing trial {trial}. Model was {model_size}MB.')
            failedTrial = createModelParametersDF([convFilters1],[convFilters2],[convFilters3],[convFilters4],
                                                  [numberOfFCLayers],[numberOfNeuronsPerFCLayer],[numberOfEpochs],
                                                  [adamLearningRate],[L2Rate],[np.nan],[np.nan])
            display(failedTrial)
        
            
        if (trial % 10 == 9): 
            modelParametersDF = createModelParametersDF(n_convFilters1,n_convFilters2,n_convFilters3,n_convFilters4,
                                                        n_FCLayers,n_NeuronsPerFCLayers,n_Epochs,adamLearningRates,L2Rates,
                                                        trainScores,devScores)
            modelParametersDF = modelParametersDF.sort_values(by='trainScore', ascending=False)
            display(modelParametersDF)
            
            top5 = modelParametersDF[0:5]
            possibleConvFilters1 = getAdjustedRange(top5['n_convFilters1'])
            possibleConvFilters2 = getAdjustedRange(top5['n_convFilters2'])
            possibleConvFilters3 = getAdjustedRange(top5['n_convFilters3'])
            possibleConvFilters4 = getAdjustedRange(top5['n_convFilters4'])
        
            possibleNumberOfFCLayers = getAdjustedRange(top5['n_FCLayers'])
            possibleNumberOfNeuronsPerFCLayer = getAdjustedRange(top5['n_NeuronsPerFCLayers'])
        
            possibleNumberOfEpochs = getAdjustedRange(top5['n_Epochs'])
            adamLearningRateCriticalPoints = calculateLogisticCriticalPoints(top5['adamLearningRates'])
            L2CriticalPoints = calculateLogisticCriticalPoints(top5['L2Rates'])

            n_convFilters1 = []
            n_convFilters2 = []
            n_convFilters3 = []
            n_convFilters4 = []
            n_FCLayers = []
            n_NeuronsPerFCLayers = []
            n_Epochs = []
            adamLearningRates = []
            L2Rates = []
            trainScores = []
            devScores = []

            if bestDevScore > 0.91:
                trial = 101
                
    displayFinalResults('../Models/best_dv_model_params.json')

In [ ]:
if __name__ == "__main__":
    main()

In non split
In split


2026-04-20 01:18:05.891318: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype float and shape [9240,200,40,1]
	 [[{{node Placeholder/_0}}]]
2026-04-20 01:18:06.199227: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz



trainScore
0.9978316426277161
devScore


2026-04-20 01:18:28.280697: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype float and shape [1680]
	 [[{{node Placeholder/_1}}]]


0.9961344599723816


2026-04-20 01:18:29.341783: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype float and shape [1680]
	 [[{{node Placeholder/_1}}]]


concluding trial  0

trainScore
0.9997334480285645
devScore
1.0
concluding trial  1

trainScore
0.6615198254585266
devScore
0.6598419547080994
concluding trial  2

trainScore
1.0
devScore
0.9993913769721985
concluding trial  3

trainScore
0.0
devScore
0.0
concluding trial  4

trainScore
0.9759572148323059
devScore
0.9653882384300232
concluding trial  5

trainScore
0.9995625019073486
devScore
0.9993913769721985
concluding trial  6

trainScore
0.6618534922599792
devScore
0.6598419547080994
concluding trial  7

trainScore
0.6619043350219727
devScore
0.6598419547080994
concluding trial  8


,n_convFilters1,n_convFilters2,n_convFilters3,n_convFilters4,n_FCLayers,n_NeuronsPerFCLayers,n_Epochs,adamLearningRates,L2Rates,trainScore,devScore
3,36,49,58,40,19,7,14,0.000210,0.046203,1.000000,0.999391
1,36,12,31,28,16,9,5,0.001239,0.039520,0.999733,1.000000
6,9,60,39,29,4,8,9,0.000141,182.505913,0.999563,0.999391
0,57,52,18,17,2,18,1,0.000105,7.068309,0.997832,0.996134
5,30,34,17,57,9,11,1,0.000125,86.326611,0.975957,0.965388
8,27,9,54,15,14,17,9,0.000344,600.268001,0.661904,0.659842
7,2,36,30,29,14,11,6,0.003786,458.314506,0.661853,0.659842
2,17,35,33,36,7,7,18,0.004719,273.476618,0.661520,0.659842
4,11,42,15,40,14,3,18,0.000308,3.466229,0.000000,0.000000


Final Model


,n_convFilters1,n_convFilters2,n_convFilters3,n_convFilters4,n_FCLayers,n_NeuronsPerFCLayers,n_Epochs,adamLearningRates,L2Rates,modelSize,trainScore,devScore,testScore
0,36,12,31,28,16,9,5,0.001239,0.03952,0.420937,0.999733,1.0,0.999391
